# Error Reproduction & Triage

When an LLM pipeline produces a bad summary in production, three questions arise:
1. **Can we reproduce it?** Stochastic models may produce different output on re-run
2. **Should a human review it?** Low-confidence outputs need escalation
3. **Is PII leaking?** Evidence documents contain sensitive information

This notebook demonstrates the SDK's answers:

| Capability | SDK Module | What It Does |
|-----------|-----------|-------------|
| Deterministic replay | `ReplayEngine` | Validates stored decisions, detects stochastic drift |
| Confidence routing | `ConfidenceRouter` | Routes low-confidence outputs to `human_review` |
| PII redaction | `Sanitizer` | Strips SSNs, phone numbers, emails before storage |
| Structured events | `emit_low_confidence`, `emit_drift_detected` | Notifies downstream systems on drift and low confidence |

> All demos run fully offline. No API keys needed.

In [ ]:
import sys, os
# Locate the criminal-evidence-workflow root (contains src/ and data/) by walking up
# from the kernel's launch dir, so this works regardless of where it was started.
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not (
    os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'data'))
):
    _root = os.path.dirname(_root)
os.chdir(_root)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import json
from pathlib import Path
from briefcase.drift import DriftCalculator
from briefcase.replay import ReplayEngine
from briefcase.sanitize import Sanitizer
from briefcase.routing import BaseRouter, RoutingDecision
from briefcase.events.emitter import emit_low_confidence, emit_drift_detected
from src.mock_llm import MockLLMProvider
from src.pipeline import summarize_report
from src.config import storage, event_bus
from src.triage import ConfidenceRouter

print("SDK modules loaded: ReplayEngine, ConfidenceRouter, Sanitizer, EventEmitter")

---
## Part 1: Stochastic Failure Detection

LLMs with `temperature > 0` are non-deterministic. The same prompt can produce different summaries on different runs. In criminal justice, this is a problem — every AI-generated summary must be reproducible.

We demonstrate this by running the **same report twice** with `temperature=0.7`. Our fixture pair (`report_001_gpt-4o_t0.7.json` and `report_001_gpt-4o_t0.7_replay.json`) simulates what happens when re-running a stochastic model: the output changes.

In [ ]:
report_text = Path("data/police_reports/report_001.txt").read_text()

# Run 1: Original output at temperature=0.7
stochastic_llm = MockLLMProvider(model="gpt-4o", temperature=0.7, simulate_latency=False)
run_1 = await summarize_report("report_001", report_text, llm=stochastic_llm)

# Run 2: "Replay" — same input, but the model produces different output
run_2 = await summarize_report("report_001", report_text, llm=stochastic_llm, replay=True)

print(f"Run 1 snapshot: {run_1['snapshot_id'][:16]}...")
print(f"Run 2 snapshot: {run_2['snapshot_id'][:16]}...")
print(f"Confidence:     {run_1['confidence']} -> {run_2['confidence']}")
print(f"Outputs match:  {run_1['summary'] == run_2['summary']}")

### Comparing the Two Outputs

The summaries are **meaningfully different** — not just one word changed. This is what makes stochastic failures dangerous: the same input produces substantively different legal summaries.

In [ ]:
print("=" * 70)
print("RUN 1 (original):")
print("=" * 70)
print(run_1["summary"][:400])
print(f"\n... (confidence: {run_1['confidence']})")

print(f"\n{'=' * 70}")
print("RUN 2 (replay — same input, different output):")
print("=" * 70)
print(run_2["summary"][:400])
print(f"\n... (confidence: {run_2['confidence']})")

### Quantifying the Drift

`DriftCalculator` measures how different the two outputs are. A high drift score means the model is unreliable at this temperature.

In [ ]:
drift_calc = DriftCalculator()
drift = drift_calc.calculate_drift([run_1["summary"], run_2["summary"]])

print(f"Drift score:       {drift.drift_score:.3f}  (0=identical, 1=completely different)")
print(f"Consistency score: {drift.consistency_score:.3f}")
print(f"Agreement rate:    {drift.agreement_rate:.3f}")

if drift.drift_score > 0.3:
    print(f"\n** HIGH DRIFT DETECTED at temperature=0.7")
    print(f"   Recommendation: Use temperature=0.0 for evidence summarization")

# Emit structured drift event
await emit_drift_detected(run_1["decision"], {
    "drift_score": drift.drift_score,
    "outputs_match": run_1["summary"] == run_2["summary"],
    "source": "stochastic_replay",
})
print(f"\n   [event] drift.detected emitted (drift_score={drift.drift_score:.3f})")

### ReplayEngine: Validating Stored Decisions

The `ReplayEngine` validates that a stored decision is internally consistent. In production with a live LLM, `replay()` would re-execute the same prompt and compare outputs.

In [ ]:
replay_engine = ReplayEngine(storage)

# Validate the original decision
replay_result = replay_engine.replay(run_1["snapshot_id"], "strict")
print(f"Replay validation:")
print(f"  Snapshot:      {run_1['snapshot_id'][:16]}...")
print(f"  Status:        {replay_result.status}")
print(f"  Outputs match: {replay_result.outputs_match}")
print(f"  Exec time:     {replay_result.execution_time_ms:.2f}ms")

---
## Part 2: Confidence-Based Routing

Not every summary should be auto-approved. The `ConfidenceRouter` inspects the `Output.confidence` on each `DecisionSnapshot` and routes decisions:

- **confidence >= 0.85** → `action="auto"` (safe to use without review)
- **confidence < 0.85** → `action="human_review"` (flagged for attorney/analyst review)

This is critical for criminal justice: the system must know when to defer to a human.

In [ ]:
router = ConfidenceRouter(confidence_threshold=0.85)

# Process all 5 reports and route each one
report_ids = ["report_001", "report_002", "report_003", "report_004", "report_005"]
llm = MockLLMProvider(model="gpt-4o", simulate_latency=False)

print(f"{'Report':<12} {'Confidence':>10} {'Action':<14} {'Reason'}")
print("─" * 70)

for rid in report_ids:
    text = Path(f"data/police_reports/{rid}.txt").read_text()
    result = await summarize_report(rid, text, llm=llm)
    
    # Router needs the stored DecisionSnapshot
    loaded = storage.load_decision(result["snapshot_id"])
    routing = await router.route(loaded)
    
    flag = " <-- FLAGGED" if routing.action == "human_review" else ""
    print(f"{rid:<12} {result['confidence']:>10.2f} {routing.action:<14} {routing.reason}{flag}")
    
    # Emit low-confidence event for flagged reports
    if routing.action == "human_review":
        await emit_low_confidence(loaded, result["confidence"], threshold=0.85)
        print(f"{'':12} {'':>10} [event] decision.low_confidence emitted")

### Why Report 003 Gets Flagged

Report 003 describes an assault with **three witnesses who contradict each other** on nearly every material fact. The model correctly reports low confidence (0.62), and the router correctly escalates it.

This is exactly the workflow Alex's team needs: when the evidence is ambiguous, the system defers to a human rather than producing a confident-sounding but unreliable summary.

In [ ]:
# Show what makes report_003 special
import json
report_003_text = Path("data/police_reports/report_003.txt").read_text()
gt = json.loads(Path("data/police_reports/report_003_ground_truth.json").read_text())

print("Conflicting witness details from ground truth:")
print()
for key, val in gt.get("conflicting_details", {}).items():
    print(f"  {key}:")
    for witness, account in val.items():
        print(f"    {witness}: {account}")
    print()

---
## Part 3: PII Sanitization

Evidence documents contain sensitive personal information — SSNs, phone numbers, email addresses. The `Sanitizer` (Rust-powered, fast) automatically redacts PII before storage.

Report 002 (DUI traffic stop) contains extensive PII that the sanitizer must catch.

In [ ]:
sanitizer = Sanitizer()

report_002 = Path("data/police_reports/report_002.txt").read_text()

# Analyze what PII is present
analysis = sanitizer.analyze_pii(report_002)
print(f"PII analysis of report_002.txt:")
print(f"  Contains PII: {sanitizer.contains_pii(report_002)}")
print(f"  Analysis:     {analysis}")

In [ ]:
# Sanitize the full document
result = sanitizer.sanitize(report_002)

print(f"Sanitization results:")
print(f"  Redactions found: {result.redaction_count}")
print(f"\n  Redaction details:")
for r in result.redactions:
    print(f"    - {r}")

In [ ]:
# Show before/after for the PII-heavy section
import re

# Find a section with PII
pii_section_start = report_002.find("SSN:")
if pii_section_start > 0:
    before = report_002[pii_section_start - 40:pii_section_start + 80]
    sanitized_text = result.sanitized
    # Find the corresponding region in sanitized text
    after_start = sanitized_text.find("[REDACTED_SSN]")
    if after_start > 0:
        after = sanitized_text[after_start - 40:after_start + 80]
    else:
        after = "(SSN fully redacted from output)"
    
    print("Before sanitization:")
    print(f"  ...{before}...")
    print(f"\nAfter sanitization:")
    print(f"  ...{after}...")

### Sanitizing JSON Decision Payloads

`sanitize_json()` processes structured data — useful for sanitizing entire `DecisionSnapshot` payloads before they're stored or transmitted.

In [ ]:
# Sanitize a decision payload
payload = {
    "inputs": [
        {"name": "document", "value": report_002[:600]},
        {"name": "query", "value": "Summarize this report"},
    ],
    "metadata": {
        "suspect_info": "Robert Fitzgerald, SSN: 478-93-6152, phone: (510) 555-4271"
    }
}

sanitized_payload = sanitizer.sanitize_json(payload)
print(f"JSON sanitization: {sanitized_payload.redaction_count} redactions")
print(f"\nSanitized metadata:")
print(f"  {sanitized_payload.sanitized.get('metadata', {})}")

## Part 4: Event Inspection

All events emitted during this notebook are collected in the `InMemoryEventBus`. In production, these would route to Slack, PagerDuty, or an audit log.

In [ ]:
print(f"Events collected: {len(event_bus.events)}\n")
for evt in event_bus.events:
    print(f"  [{evt.event_type}]")
    print(f"    Decision: {evt.decision_id}")
    print(f"    Time:     {evt.timestamp.isoformat()}")
    print(f"    Payload:  {evt.payload}")
    print()

## Key Takeaways

- **Stochastic failures are real**: Temperature=0.7 produces meaningfully different summaries on re-run (drift score: ~0.69). For criminal evidence, use temperature=0.0.
- **ReplayEngine validates decisions**: Stored snapshots can be replayed and validated for consistency
- **Confidence routing works**: `ConfidenceRouter` correctly flags report_003 (conflicting witnesses, conf=0.62) for human review while auto-approving clear cases
- **PII is handled at the SDK level**: Rust-powered `Sanitizer` catches SSNs, phones, and emails — no regex maintenance needed
- **Structured event emission**: `emit_drift_detected` and `emit_low_confidence` fire automatically during triage, collected in the event bus for downstream processing
- **Defense in depth**: Replay catches model instability, routing catches low confidence, sanitization catches data leakage, events notify downstream systems — four independent safety layers

**Next:** See [04_data_lineage_and_compliance.ipynb](./04_data_lineage_and_compliance.ipynb) for immutable data versioning and compliance-grade lineage.